# IMDB Sentiment Analysis (LSTM)

This notebook performs end-to-end sentiment classification on the IMDB dataset using a simple LSTM model. The logic is unchanged from the original version; only formatting, commentary, and section organization were enhanced for readability and a refreshed visual style.

---


In [ ]:
# === Core Library Imports ===
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import ModelCheckpoint

# Load raw dataset and preview first rows
data = pd.read_csv('dataset_imdb.csv')
print("Raw sample:")
print(data.head())

Raw sample:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


### 1. Library Imports & Raw Data Preview
Short cell to load dependencies and peek at the raw dataset.


In [4]:
# Stopword set (loaded once)
english_stops = set(stopwords.words('english'))

# Preprocessing & label encoding pipeline
# Returns token lists (not yet sequenced) and binary targets
def load_dataset():
    df = pd.read_csv('dataset_imdb.csv')
    x_data = df['review']      # textual reviews
    y_data = df['sentiment']   # class labels

    # Strip HTML fragments
    x_data = x_data.replace({'<.*?>': ''}, regex=True)
    # Purge non-letter characters
    x_data = x_data.replace({'[^A-Za-z]': ' '}, regex=True)
    # Token filter: remove stopwords
    x_data = x_data.apply(lambda review: [w for w in review.split() if w not in english_stops])
    # Normalize case
    x_data = x_data.apply(lambda review: [w.lower() for w in review])
    # Sentiment → binary flag
    y_data = y_data.replace({'positive': 1, 'negative': 0})
    return x_data, y_data

# Execute preprocessing
x_data, y_data = load_dataset()
print("Sample token lists:\n", x_data.head())
print("\nSample targets:\n", y_data.head())

Sample token lists:
 0    [one, reviewers, mentioned, watching, oz, epis...
1    [a, wonderful, little, production, the, filmin...
2    [i, thought, wonderful, way, spend, time, hot,...
3    [basically, family, little, boy, jake, thinks,...
4    [petter, mattei, love, time, money, visually, ...
Name: review, dtype: object

Sample targets:
 0    1
1    1
2    1
3    0
4    1
Name: sentiment, dtype: int64


### 2. Text Cleaning & Stopword Handling
Defines the dataset loader and lightweight preprocessing pipeline.


In [5]:
# Partition dataset (80/20)
x_train, x_test, y_train, y_test = train_test_split(
    x_data, y_data, test_size=0.2, random_state=42
)
print(f"Train samples: {len(x_train)} | Test samples: {len(x_test)}")

Train samples: 40000 | Test samples: 10000


### 3. Train/Test Split
Standard 80/20 stratification (random_state fixed for reproducibility).


In [6]:
# Determine representative sequence length (mean length, ceiled)
def get_max_length():
    lengths = []
    for review in x_train:
        lengths.append(len(review))
    return int(np.ceil(np.mean(lengths)))

max_length = get_max_length()
print("Chosen padding length:", max_length)

Chosen padding length: 130


### 4. Sequence Length Heuristic
Average review length used as padding length (ceiled).


In [7]:
# Fit tokenizer on training corpus only
token = Tokenizer(lower=False)  # Already lowered earlier
token.fit_on_texts(x_train)

# Map tokens to integer ids
x_train = token.texts_to_sequences(x_train)
x_test = token.texts_to_sequences(x_test)

# Uniform length padding
x_train = pad_sequences(x_train, maxlen=max_length, padding='post', truncating='post')
x_test = pad_sequences(x_test, maxlen=max_length, padding='post', truncating='post')

# Vocabulary size (+1 for padding slot)
total_words = len(token.word_index) + 1
print("Vocab size:", total_words)
print("Sample encoded review:", x_train[0][:20], "...")

Vocab size: 92546
Sample encoded review: [  145     1   702  2078    38  1815  1945  4247  6378   698  4753 21764
   135     2  6062    22   680    30     5  1922] ...


### 5. Tokenization & Padding
Fit tokenizer on training data only; pad/truncate to uniform length.


In [8]:
# Build LSTM classifier
EMBED_DIM = 32
LSTM_OUT = 64
model = Sequential([
    Embedding(total_words, EMBED_DIM, input_length=max_length),
    LSTM(LSTM_OUT),
    Dense(1, activation='sigmoid')  # Binary sentiment
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# Ensure model graph initialized for summary
model.build(input_shape=(None, max_length))
model.summary()

c:\Users\vivek\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 130, 32)        │     2,961,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,986,369 (11.39 MB)

 Trainable params: 2,986,369 (11.39 MB)

 Non-trainable params: 0 (0.00 B)

### 6. Model Definition
Minimal Embedding + LSTM + Sigmoid classifier.


In [9]:
# Model checkpoint (keep best by accuracy)
checkpoint = ModelCheckpoint(
    'models/LSTM.keras',
    monitor='accuracy',
    save_best_only=True,
    verbose=1
)
# Train
history = model.fit(
    x_train, y_train,
    batch_size=128,
    epochs=5,
    callbacks=[checkpoint]
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.5426 - loss: 0.6786
Epoch 1: accuracy improved from -inf to 0.57500, saving model to models/LSTM.keras

Epoch 1: accuracy improved from -inf to 0.57500, saving model to models/LSTM.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 58s 177ms/step - accuracy: 0.5427 - loss: 0.6785
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 58s 177ms/step - accuracy: 0.5427 - loss: 0.6785
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 0.5033 - loss: 0.6933
Epoch 2: accuracy did not improve from 0.57500
313/313 ━━━━━━━━━━━━━━━━━━━━ 63s 201ms/step - accuracy: 0.5033 - loss: 0.6933
Epoch 3/5

Epoch 2: accuracy did not improve from 0.57500
313/313 ━━━━━━━━━━━━━━━━━━━━ 63s 201ms/step - accuracy: 0.5033 - loss: 0.6933
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.5290 - loss: 0.6892
Epoch 3: accuracy did not improve from 0.57500
313/313 ━━━━━━━━━━━━━━━━━━━━ 61s 197ms/step - accuracy: 0.5290 - loss: 0.6892
Epoch 4/5

Epoch 3: 

### 7. Training & Checkpointing
Saves the best-performing weights (monitoring accuracy) during training.


In [10]:
# Threshold predictions at 0.5
y_pred = (model.predict(x_test, batch_size=128) > 0.5).astype("int32")
# Manual accuracy computation
correct = np.sum(y_test.values == y_pred.flatten())
wrong = len(y_pred) - correct
acc = correct / len(y_pred) * 100
print(f"Correct: {correct} | Wrong: {wrong} | Accuracy: {acc:.2f}%")

79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step
Correct: 7955 | Wrong: 2045 | Accuracy: 79.55%
Correct: 7955 | Wrong: 2045 | Accuracy: 79.55%


### 8. Evaluation on Held-Out Test Set
Manual accuracy computation after thresholding predictions.


In [11]:
# Reload best checkpoint
loaded_model = load_model('models/LSTM.keras')

# User input
review = str(input("Enter a movie review: "))

# Basic clean (remove punctuation/symbols)
regex = re.compile(r'[^a-zA-Z\s]')
review = regex.sub('', review)
print("Cleaned:", review)

# Stopword filtering
words = review.split()
filtered = [w for w in words if w not in english_stops]
filtered_text = ' '.join(filtered).lower()
print("Filtered:", filtered_text)

# Encode & pad
encoded = token.texts_to_sequences([filtered_text])
encoded = pad_sequences(encoded, maxlen=max_length, padding='post', truncating='post')
print("Encoded vector (truncated view):", encoded[0][:25], '...')

# Predict sentiment
score = loaded_model.predict(encoded)[0][0]
label = "Positive" if score >= 0.7 else "Negative"
print(f"Predicted Sentiment: {label} (score={score:.3f})")

Cleaned: A groundbreaking scifi film that blends action and philosophical ideas though its concepts can be complex
Filtered: a groundbreaking scifi film blends action philosophical ideas though concepts complex
Encoded vector (truncated view): [  39 7087 5176    4 8997  112 4159  923   68 5182 1212    0    0    0
    0    0    0    0    0    0    0    0    0    0    0] ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 781ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 781ms/step
Predicted Sentiment: Positive (score=0.746)
Predicted Sentiment: Positive (score=0.746)
